# Quantum kernel matrices in Qiskit
**Quantum Machine Learning · Week 2 (Quantum kernels) · Day 2**

Lesson page: https://sara-aminpour.com/tutorials/qml/w2d2.html

In [ ]:
# On Google Colab, this installs qiskit (about a minute). On your own computer, skip it and
# install the packages once instead: pip install qiskit==2.5.2 scikit-learn torch matplotlib
import sys
if "google.colab" in sys.modules:
    %pip install --quiet qiskit==2.5.2

A quantum-kernel SVM needs a whole **matrix** of kernel values: training points against training
points, and test points against training points. On hardware, every entry is its own circuit, run for
many shots. Today: building the matrices properly, counting the bill, and what shot noise does.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit.primitives import StatevectorSampler
from sklearn.datasets import make_moons
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

X, y = make_moons(n_samples=90, noise=0.15, random_state=0)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=60, random_state=0)
scaler = MinMaxScaler((0, np.pi / 2)).fit(X_train)       # fit on the training points only
X_train, X_test = scaler.transform(X_train), scaler.transform(X_test)

def zz_map(x, reps=2):
    qc = QuantumCircuit(2)
    for _ in range(reps):
        qc.h([0, 1])
        qc.p(2 * x[0], 0)
        qc.p(2 * x[1], 1)
        qc.cx(0, 1)
        qc.p(2 * (np.pi - x[0]) * (np.pi - x[1]), 1)
        qc.cx(0, 1)
    return qc

def statevectors(data):
    return np.array([Statevector(zz_map(x)).data for x in data])

S_train, S_test = statevectors(X_train), statevectors(X_test)
K_train = np.abs(S_train.conj() @ S_train.T) ** 2        # 30 x 30: training against training
K_test = np.abs(S_test.conj() @ S_train.T) ** 2          # 60 x 30: test against TRAINING points
print("K_train", K_train.shape, " K_test", K_test.shape)
n, m = len(X_train), len(X_test)
print(f"circuits on hardware: {n * (n - 1) // 2} for K_train (diagonal = 1, symmetric) + {m * n} for K_test")

## Estimating the matrix from shots
Each entry is an inversion test: the fraction of shots that read 00. Here are all 435 training circuits,
with 1,000 shots each.

In [ ]:
pairs = [(i, j) for i in range(n) for j in range(i + 1, n)]
circuits = []
for i, j in pairs:
    qc = zz_map(X_train[i]).compose(zz_map(X_train[j]).inverse())
    qc.measure_all()
    circuits.append(qc)
results = StatevectorSampler(seed=1).run(circuits, shots=1000).result()

K_shots = np.eye(n)
for (i, j), result in zip(pairs, results):
    K_shots[i, j] = K_shots[j, i] = result.data.meas.get_counts().get("00", 0) / 1000
print(f"largest error of any entry: {np.abs(K_shots - K_train).max():.3f}")

eig_exact = np.linalg.eigvalsh(K_train)
eig_shots = np.linalg.eigvalsh(K_shots)
print(f"smallest eigenvalue: exact {eig_exact.min():.1e}, from shots {eig_shots.min():.3f}")
print(f"negative eigenvalues: exact {np.sum(eig_exact < -1e-9)}, from shots {np.sum(eig_shots < -1e-9)}")

plt.plot(np.sort(eig_exact)[::-1], "o-", ms=4, color="#7b5cfa", label="exact")
plt.plot(np.sort(eig_shots)[::-1], "o-", ms=4, color="#ff7a59", label="1,000 shots per entry")
plt.axhline(0, color="#2c2842", lw=1)
plt.yscale("symlog", linthresh=1e-3)
plt.xlabel("eigenvalue number")
plt.ylabel("eigenvalue")
plt.legend()
plt.show()

Only about 16 eigenvalues of the exact matrix are clearly positive: two qubits give density matrices with
16 numbers, so the matrix has rank 16 at most. Shot noise nudges the zero eigenvalues both ways, and the
negative ones mean the estimate is no longer a valid kernel.

## Repair: clip the negative eigenvalues
Replace them by zero and rebuild the matrix: the closest valid kernel matrix.

In [ ]:
def clip_to_psd(K):
    values, vectors = np.linalg.eigh(K)
    return (vectors * np.clip(values, 0, None)) @ vectors.T

K_fixed = clip_to_psd(K_shots)
print("negative eigenvalues after the repair:", np.sum(np.linalg.eigvalsh(K_fixed) < -1e-9))
for name, K in [("exact", K_train), ("from shots", K_shots), ("from shots, repaired", K_fixed)]:
    model = SVC(kernel="precomputed").fit(K, y_train)
    print(f"{name:22s} test accuracy {model.score(K_test, y_test):.3f}")

## Your turn: how many shots does the SVM need?
Shots follow a binomial distribution, so we can simulate them fast: `rng.binomial(shots, K) / shots`.
For 10, 30, 100, 1,000 and 10,000 shots per entry, estimate both matrices (keep the training matrix symmetric
with ones on the diagonal), repair the training matrix, and record the test accuracy averaged over 10 repetitions.

In [ ]:
rng = np.random.default_rng(0)

def noisy(K, shots, symmetric):
    E = rng.binomial(shots, np.clip(K, 0, 1)) / shots
    if symmetric:
        E = np.triu(E, 1)
        E = E + E.T
        np.fill_diagonal(E, 1.0)
    return E

for shots in [10, 30, 100, 1000, 10000]:
    accuracies = []
    for repetition in range(10):
        pass     # TODO: noisy training and test matrices, repair the training one, fit SVC, append the test accuracy
    print(shots, np.mean(accuracies) if accuracies else None)

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
rng = np.random.default_rng(0)
for shots in [10, 30, 100, 1000, 10000]:
    accuracies = []
    for repetition in range(10):
        K_tr = clip_to_psd(noisy(K_train, shots, True))
        K_te = noisy(K_test, shots, False)
        accuracies.append(SVC(kernel="precomputed").fit(K_tr, y_train).score(K_te, y_test))
    print(f"{shots:6d} shots per entry: test accuracy {np.mean(accuracies):.3f}")
print("On this small, well-scaled problem the SVM barely notices the shot noise: kernel values are large")
print("and well separated. Thursday shows where that breaks: when every kernel value becomes tiny.")